# CC-MMD 2026 — Few-Shot CoT MAMI (Western) Evaluation
Few-shot misogyny classification on Western/English memes — CoT prompt with cultural
and geolocation context, grounded with real labelled examples.

**Backend:** HuggingFace (Flash Attention 2 + batched query inference)

### Description
This notebook evaluates `Qwen/Qwen2.5-VL-7B-Instruct` on the **MAMI (Western)** test set.
Runs Indian and Chinese cultural perception partitions.

**Dataset:** MAMI — Western/English memes (`data/MAMI/`)
**Partitions:** `mami_indian` · `mami_chinese`
**Model:** `Qwen/Qwen2.5-VL-7B-Instruct`

In [34]:
# Install dependencies
!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q 'qwen-vl-utils[decord]==0.0.8'
!pip install -q pandas tqdm Pillow scikit-learn
!pip install -q flash-attn --no-build-isolation
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [35]:
# ── Option A: Mount Google Drive ──────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = "/content/drive/MyDrive/cc_mmd_dataset"

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/data/cc_mmd_dataset"


MAMI_DIR  = "/content/drive/MyDrive/data/MAMI"   # test images + test.csv
OUTPUT_DIR = "/content/results"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:

# ── API Keys from Colab Secrets ───────────────────────────────────────────────
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""

print("HF token  :", "set" if os.environ["HF_TOKEN"] else "NOT SET")


HF token  : set


In [37]:
# ── CONFIG — edit this cell before running ────────────────────────────────────

SELECTED_MODELS = [
    "Qwen/Qwen2.5-VL-7B-Instruct",
]

# MAMI (Western) partitions only
RUN_PARTITIONS = [
    "mami_indian",   # Indian perception of Western memesß
    #"mami_chinese",  # Chinese perception of Western memes
]

BATCH_SIZE = 2

print(f"Models     : {SELECTED_MODELS}")
print(f"Partitions : {RUN_PARTITIONS}")
print(f"Batch size : {BATCH_SIZE}")

Models     : ['Qwen/Qwen2.5-VL-7B-Instruct']
Partitions : ['mami_indian']
Batch size : 2


In [38]:
# ── Few-shot examples ────────────────────────────────────────────────────────
# Each entry: image path, ground-truth label (0/1), classification string, explanation.
# Drawn from the dev split so the model sees representative real examples.
# To change examples, edit the list below.

INDIAN_FEW_SHOT = [{
          "image_path":  f"{BASE_DIR}/MDMD/dev/1110.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Two-panel Tamil meme. Top: Tamil text advising that when unsure what to cook, "
              "open a container for ideas. Bottom: Tamil comedian (Vadivelu) looking at a container "
              "labelled 'இல்லத்தரசிகள்' (Housewives). Hashtag #ரவை (Rava/Semolina) visible. "
              "The audience is explicitly addressed as housewives."
          ),
          "explanation": (
              "The meme frames housewives as the audience for a cooking tip, explicitly labelling "
              "women as 'இல்லத்தரசிகள்' (Housewives) and reducing their identity to that of "
              "domestic cooks. It reinforces the gender-based assumption that cooking and "
              "domestic duties are a woman's primary identity and responsibility."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/1096.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Two-panel Tamil meme. Top panel: comedian actor (Vadivelu) labelled '*singles' "
              "with text 'Oru poi yavathu sol kannae.. Kanae' (At least tell one lie, dear). "
              "Bottom panel: group of men from a Tamil film labelled '*Love failures' "
              "with text 'Ava solrathu ellame poi than' (Everything she says is a lie). "
              "Contrasting expressions — longing vs. bitter."
          ),
          "explanation": (
              "The meme contrasts single men longing for any female attention with 'love failure' "
              "men who claim women are congenital liars. By framing all women in romantic contexts "
              "as inherently deceptive, the meme reinforces a misogynistic stereotype that women "
              "manipulate and lie to men, targeting women's trustworthiness based solely on gender."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/1250.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Illustrated cartoon meme. Standing man gesturing in frustration toward a "
              "seated woman eating messily from a bowl on a sofa, dirty dishes piled on a side table. "
              "Tamil text at top: 'Love is just suffering… Marriage is suffering untold…' "
              "Way2news app watermark at bottom. Cartoon art style with exaggerated expressions."
          ),
          "explanation": (
              "The illustration explicitly frames the wife as lazy and negligent — eating carelessly "
              "while dishes pile up — and the husband as exasperated. Paired with text equating "
              "marriage to 'untold suffering,' the meme implies women cause marital unhappiness "
              "through domestic negligence, reinforcing the misogynistic stereotype of wives as "
              "burdens who ruin domestic life after marriage."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/1440.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Real photograph of an Indian bride on her wedding day. She wears multiple "
              "heavy gold necklaces and bridal jewellery that cover almost the entire neck and chest. "
              "Tamil text overlay: 'Due to lack of space to tie the mangalsutra (wedding necklace), "
              "the marriage is temporarily postponed.' No other visual elements."
          ),
          "explanation": (
              "The meme uses the bride's photograph to mock the idea that excessive jewellery "
              "(implying the woman's large body) leaves no room for the wedding mangalsutra, "
              "framing her physical appearance as a comedic barrier to marriage. "
              "This is targeted body-shaming of a real woman on her wedding day, reinforcing the "
              "harmful idea that a woman's body is a valid subject of public ridicule."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/1234.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Four-panel '90s PARITHABANGAL' (90s jokes) comparison meme. Top-left: Tamil "
              "actor (Jayam Ravi) pointing. Top-right: Dora the Explorer cartoon character. "
              "Bottom-left: Tamil comedian with glasses pointing. Bottom-right: Real woman in a "
              "saree walking on a road. Text repeated in both rows: 'இவ தான் ஊர் சுத்தி' "
              "(She is the one who roams the neighbourhood)."
          ),
          "explanation": (
              "'ஊர் சுத்தி' (neighbourhood roamer) is a Tamil phrase carrying negative/sexually "
              "derogatory connotations when applied to women, implying promiscuity or social "
              "impropriety. The meme equates Dora the Explorer's innocent adventures with a real "
              "woman's public movement, slut-shaming her simply for being seen walking in public. "
              "This is misogynistic ridicule that polices women's freedom of movement."
          ),
      },

      # ── NOT-MISOGYNY examples (5) ──────────────────────────────────────────

      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/1006.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Four-panel 'Practical Exam's Be Like' meme. Top-left: relaxed external "
              "examiner with snacks on table. Top-right: confident student. Caption: 'Avangalam "
              "Kooda Paravala...' (No problem with them). Bottom-left: stern internal staff member. "
              "Bottom-right: nervous student. Caption: 'Ungala Patha Than Romba Bayama Iruku...' "
              "(Seeing you is very scary). Minion Memes Tamil watermark."
          ),
          "explanation": (
              "The meme humorously contrasts a student's relaxed attitude toward external examiners "
              "with their fear of internal staff during practical exams. The humour targets the "
              "exam system and institutional authority universally, applicable to all students "
              "regardless of gender. No gender-based stereotyping, objectification, or "
              "discrimination is present."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/360.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Facebook post by 'Rahman Mc'. Text: 'me finally trying to sleep peacefully' "
              "(sleep emoji) / mosquito emoji 'in my ears'. Below: a strip of three close-up "
              "reaction shots of a young woman making animated expressions — surprised, grinning, "
              "waving — captioned 'Hey unnathan..unnathaannn' (playfully calling out, mimicking "
              "the mosquito's buzz). Personal social media format."
          ),
          "explanation": (
              "The meme uses a woman's animated facial expressions to personify the annoying "
              "mosquito that disrupts sleep — a universally relatable bedtime experience. "
              "The humour is situational and directed at a common everyday nuisance; the woman "
              "is used as a comedic prop for the mosquito, not targeted based on gender. "
              "No stereotyping, objectification, or misogynistic content is present."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/656.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Dark-background text meme with Tamil movie still below. Top text: 'Nanae "
              "ennaikachum dhan padikilam nu mudivu edukuren / Apo nu pathu slow va irukiyae da' "
              "(I keep deciding I'll study someday / Then looking at how slow [I am]). Movie still "
              "shows a man looking at a woman in traditional dress, labelled '*My laptop'. "
              "Vera Level Memes / _dharshan_ watermark."
          ),
          "explanation": (
              "The meme uses a Tamil movie still to personify the student's slow and sluggish "
              "laptop, expressing relatable frustration about procrastination and technology. "
              "The humour applies equally to all students regardless of gender; the woman in the "
              "movie clip is used as the laptop metaphor, not targeted as a gender stereotype. "
              "No discriminatory or misogynistic content is present."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/214.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Dark-themed Facebook post by 'Sanjay Krishna'. Text: 'My mood nowadays, "
              "when the topic is about Indian Cricket Team...' Below: Tamil film still showing "
              "two men in a tense confrontation, subtitle 'Neenga ennamo pesunga, antha pnuda "
              "team paththi enta kekaathinga...' (You all keep talking, but don't ask me about "
              "that team...). GOLDMINES entertainment watermark."
          ),
          "explanation": (
              "The meme expresses comedic frustration at the Indian cricket team's performance "
              "through a Tamil movie dialogue reference, targeting sports culture and fan "
              "sentiment universally. No women are referenced, targeted, or stereotyped. "
              "The humour is entirely about sports and has no gender-based dimension."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/MDMD/dev/787.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Multi-panel collage meme. Top row: photographs of APJ Abdul Kalam, cricketer "
              "Virender Sehwag, Amitabh Bachchan. Middle row: two South Indian actresses, Mahatma "
              "Gandhi. Text banner: 'Bold and Beautiful People are born in OCTOBER'. Bottom panel: "
              "Tamil movie still of a young man boasting to friends, captioned '** oct born — "
              "Bold & beautiful ahh nanu' (I am bold and beautiful — I'm October born). "
              "SK_Mc / Mokka Postu watermark."
          ),
          "explanation": (
              "The meme is a light-hearted birthday-month pride meme celebrating October-born "
              "people across genders, featuring both male and female public figures equally as "
              "positive exemplars. The self-congratulatory humour is inclusive and contains no "
              "gender-based targeting, stereotyping, objectification, or discrimination."
          ),
      },
  ]


CHINESE_FEW_SHOT = [

      # ── MISOGYNY examples (5) ──────────────────────────────────────────────

      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/204.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Chinese ink-style illustrated meme. Two plump traditional female figures "
              "in red robes and hair buns. One stands with her back turned saying 'EN EN' "
              "(agreeing sounds); the other faces her saying '听我说' (Listen to me). "
              "Text above: '孩子不听话是你不对 / 孩子学习成绩不好 / 还是你不对' "
              "(If the child misbehaves it's your fault / If the child's grades are poor / "
              "it's still your fault). 悟家族 watermark."
          ),
          "explanation": (
              "The meme places sole blame on the mother for a child's misbehaviour and poor "
              "academic performance, with no mention of the father's responsibility. "
              "This perpetuates the Chinese cultural stereotype that child-rearing failures "
              "fall exclusively on the mother, reflecting gender-based discrimination rooted "
              "in patriarchal family structures where women bear the entire domestic burden."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/1303.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Minimalist text-card meme in beige and orange. Large title in orange: "
              "'一定是女的' (Must be a woman) with pinyin romanisation above. "
              "Section header '表现：' (Signs/Manifestations) followed by three bullet quotes: "
              "'理科女生学的不如男生' (STEM girls learn worse than boys) / "
              "'游戏打这么菜一定是女的' (Playing games this badly must be a woman) / "
              "'车开这么差一定是女司机' (Driving this poorly must be a female driver). "
              "听她说《重塑》(RESHAPE) branding at bottom."
          ),
          "explanation": (
              "The meme uses 'must be a woman' as the punchline for three separate domains of "
              "incompetence — academics, gaming, and driving. It explicitly treats being female "
              "as sufficient explanation for poor performance, perpetuating harmful gender "
              "stereotypes that women are inherently less capable than men across multiple fields. "
              "This is direct gender-based discrimination regardless of its list format."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/932.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Hand-drawn illustration meme. Three figures seated at a table: older girl "
              "(left, looking upset), adult woman/mother (centre), younger boy (right, happily "
              "eating fries). Title text: '只因为我是姐姐 / 我就要学会「谦让」/ "
              "但其实我也很想多吃几根薯条' (Just because I'm the older sister / I must learn "
              "to 'yield' / but I also want more fries). Mother's speech bubble: "
              "'我买了一份薯条，你和弟弟分着吃' (I bought one portion of fries — "
              "share it with your brother)."
          ),
          "explanation": (
              "The meme shows an older sister told to yield her food to a younger brother "
              "purely because she is the girl in the family. The use of 'just because I'm "
              "the older sister' directly names gender as the reason for unequal resource "
              "distribution. This depicts institutionalised gender discrimination in Chinese "
              "family structures where daughters are systematically expected to sacrifice "
              "for sons, a form of son-preference misogyny."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/1589.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Text-only meme on a plain white background. Bold cascading text in "
              "black and red: '不学习就考不上 / 考不上就要嫁人 / 两年生三 / 老公不爱 / "
              "婆婆虐待 / 洗衣扫地 / 做饭买菜' (Don't study → fail exams / Fail exams → "
              "must get married / Give birth 3 times in 2 years / Husband won't love you / "
              "Mother-in-law abuses you / Wash clothes, sweep floors / Cook and shop). "
              "'考不上' and '嫁人' are highlighted in red for emphasis."
          ),
          "explanation": (
              "The meme presents a grim chain of consequences exclusively for girls: failure "
              "in education leads directly to forced marriage, followed by childbirth, "
              "domestic servitude, and abuse. By framing marriage as a punishment and "
              "reducing a woman's future to household labour and suffering, the meme "
              "reinforces deeply misogynistic expectations that a girl's worth is tied "
              "to academic performance and that domestic oppression is her inevitable fate "
              "without it."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/1656.jpg",
          "label":       1,
          "classification": "misogyny",
          "scene_graph": (
              "Scene: Minimalist text-card meme in beige and orange (same design system as 1303). "
              "Large title: '外貌攻击' (Appearance Attack) with pinyin above. "
              "Section header '表现：' (Signs) with four quoted examples: "
              "'头发长见识短' (Long hair, short on knowledge) / "
              "'你怎么长这么黑' (How did you get so dark) / "
              "'腿这么粗还要穿短裙' (Legs so thick still wearing a short skirt) / "
              "'长成这样，再厉害也没有用吧' (Looking like this, capability is useless). "
              "听她说《重塑》branding at bottom."
          ),
          "explanation": (
              "The meme catalogues appearance-based attacks directed at women: intelligence "
              "dismissed via the sexist proverb 'long hair, short on knowledge,' skin tone "
              "shaming, body shaming about leg size and clothing choice, and the claim that "
              "a woman's competence is worthless if she is not physically attractive. "
              "Every example targets women specifically and frames their physical appearance "
              "as grounds for dismissal or ridicule, constituting clear gender-based "
              "body-shaming and misogyny."
          ),
      },

      # ── NOT-MISOGYNY examples (5) ──────────────────────────────────────────

      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/423.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Photo of two black-and-white cats with exaggerated facial expressions. "
              "One cat holds a hand of playing cards. Both cats look wide-eyed and comically "
              "stressed. Text overlay: '像我的生活...' (Like my life...) / "
              "'总是差一点就顺呐' (Always just a little short of going smoothly)."
          ),
          "explanation": (
              "The meme uses funny cat expressions to express universal frustration about "
              "life never going quite right. The humour is relatable and directed at the "
              "shared human experience of near-misses and bad luck. No gender targeting, "
              "no stereotyping, and no misogynistic content of any kind is present."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/1211.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Two-panel cat interview meme. Top panel: tabby cat sitting at a desk "
              "being interviewed with a TV microphone, text: '我觉得找工作不一定要找工资高的 / "
              "但一定要找自己喜欢的！' (I think you don't need a high-paying job / but find "
              "one you love!). Bottom panel: same cat looking sheepish, text: "
              "'那你喜欢什么样的？/ 我喜欢工资高的' (So what kind do you like? / "
              "I like high-paying ones)."
          ),
          "explanation": (
              "The meme is a self-contradictory humour piece about job-hunting — the speaker "
              "preaches passion over salary then immediately admits they want a high salary. "
              "The joke targets universal workplace attitudes and self-deception, applying "
              "equally to all genders. No women are targeted, stereotyped, or demeaned."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/1403.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Office photo. A person sits with their back to the camera at a computer "
              "desk. A printed meme is taped to the back of their chair reading: "
              "'国庆7天+中秋3天 / 调休完等于总共放4天' (National Day 7 days + Mid-Autumn 3 days / "
              "After make-up work days = total 4 days off) / joke comparing it to the "
              "monkey-counting riddle where 7+3=4. A screenshot of a person at a computer "
              "is embedded in the printed note."
          ),
          "explanation": (
              "The meme jokes about the Chinese 调休 (compensatory work-day) system that "
              "converts long holiday periods into effectively short ones — a universally "
              "shared workplace frustration. The humour is directed at the national holiday "
              "scheduling policy and applies to all workers regardless of gender. "
              "No misogynistic content is present."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/1238.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Two-panel cartoon dog meme. Top panel: cute yellow cartoon dog sitting "
              "atop a large pile of papers and documents, smiling calmly. Text: '大难临头' "
              "(When disaster is upon you). Bottom panel: same dog cheerfully folding the "
              "documents into paper planes. Text: '先玩一会儿' (Play for a bit first). "
              "Bright, cheerful animation style throughout."
          ),
          "explanation": (
              "The meme humorously captures the universal human tendency to procrastinate "
              "when facing an overwhelming workload — turning crisis documents into toys. "
              "The joke about avoidance behaviour applies to everyone regardless of gender. "
              "No women are referenced, targeted, or stereotyped; the content is "
              "entirely gender-neutral."
          ),
      },
      {
          "image_path":  f"{BASE_DIR}/CMMD/dev/544.jpg",
          "label":       0,
          "classification": "not-misogyny",
          "scene_graph": (
              "Scene: Two-panel white Shiba Inu dog meme with soft sky background. "
              "Top panel: dog with tearful eyes, text: '以前：为什么不喜欢我' "
              "(Before: Why don't you like me). "
              "Bottom panel: same dog with a confident, dismissive expression, text: "
              "'现在：没品味的东西' (Now: You have no taste). "
              "Gentle pastel sky backdrop, no other visual elements."
          ),
          "explanation": (
              "The meme expresses the relatable emotional arc from past insecurity about "
              "rejection to present self-confidence — a universal glow-up sentiment with "
              "no gender-specific framing. The dog character is gender-neutral and the "
              "humour targets shared emotional experiences. No women are targeted, "
              "stereotyped, objectified, or demeaned."
          ),
      },
  ]


 # ── Partition-to-few-shot mapping ─────────────────────────────────────────────
PARTITION_FEW_SHOT = {
    "mami_indian":   INDIAN_FEW_SHOT,
    "mami_chinese":  CHINESE_FEW_SHOT,
    "mdmd_original": INDIAN_FEW_SHOT,
    "mdmd_irish":    INDIAN_FEW_SHOT,
    "mdmd_chinese":  INDIAN_FEW_SHOT,
    "cmmd_original": CHINESE_FEW_SHOT,
    "cmmd_irish":    CHINESE_FEW_SHOT,
    "cmmd_indian":   CHINESE_FEW_SHOT,
}

# Quick validation
print(f"Indian few-shot : {len(INDIAN_FEW_SHOT)} examples "
      f"({sum(1 for e in INDIAN_FEW_SHOT if e['label']==1)}M / "
      f"{sum(1 for e in INDIAN_FEW_SHOT if e['label']==0)}NM)")
print(f"Chinese few-shot: {len(CHINESE_FEW_SHOT)} examples "
      f"({sum(1 for e in CHINESE_FEW_SHOT if e['label']==1)}M / "
      f"{sum(1 for e in CHINESE_FEW_SHOT if e['label']==0)}NM)")
for partition, examples in PARTITION_FEW_SHOT.items():
    print(f"  {partition:20s} → {len(examples)} examples")

Indian few-shot : 10 examples (5M / 5NM)
Chinese few-shot: 10 examples (5M / 5NM)
  mami_indian          → 10 examples
  mami_chinese         → 10 examples
  mdmd_original        → 10 examples
  mdmd_irish           → 10 examples
  mdmd_chinese         → 10 examples
  cmmd_original        → 10 examples
  cmmd_irish           → 10 examples
  cmmd_indian          → 10 examples


In [39]:

ALL_PARTITIONS = {
    # MAMI — Western/English memes (Indian + Chinese perspectives)
    "mami_indian": {
        "image_dir":   MAMI_DIR,
        "csv":         f"{MAMI_DIR}/test_with_labels_mami.csv",  # transcriptions only, no labels
        "country":     "India",
        "language":    "English",
        "label_col":   "indian_labels",
        "description": "MAMI — Indian perception (test)",
        "is_cross_culture": True
    },
    "mami_chinese": {
        "image_dir":   MAMI_DIR,
        "csv":         f"{MAMI_DIR}/test_with_labels_mami.csv",  # transcriptions only, no labels
        "country":     "China",
        "language":    "English",
        "label_col":   "chinese_labels",
        "description": "MAMI — Chinese perception (test)",
        "is_cross_culture": True
    },
}

PARTITIONS = {k: ALL_PARTITIONS[k] for k in RUN_PARTITIONS if k in ALL_PARTITIONS}
print(f"Running {len(PARTITIONS)} partition(s): {list(PARTITIONS.keys())}")

Running 1 partition(s): ['mami_indian']


In [40]:
# ── Prompt + utilities ────────────────────────────────────────────────────────
import json
import base64
import glob
import csv as csv_module
import pandas as pd
from tqdm import tqdm
from PIL import Image
from io import BytesIO
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


NATIVE_PROMPT_TEMPLATE = """
You are a highly advanced misogyny detector, designed to analyse visual scenes
and evaluate memes for misogynistic content by examining both visual and
textual components, interpreted within the cultural, social, and linguistic
context of {country}.

You will be provided with a {language} meme image.

Step-by-Step Instructions:
1. Generate a scene graph by identifying all entities, objects, attributes,
   and relationships in the scene. Include characters, expressions, text
   overlays, symbols, layout, and background elements.
2. Use the scene graph from step 1 to analyse the visual elements and evaluate
   them against the socio-cultural context of {country}.

Classification Criteria:
- Classify as "misogyny" if the content directly or indirectly targets women
  or girls through: stereotyping, hatred, discrimination, sexualization,
  objectification, gender-based violence, insults, victim-blaming, mockery,
  or trivialization — even if conveyed through sarcasm or humor.
- Classify as "not-misogyny" if none of the above criteria apply.

Rules:
- Be objective, culturally aware, and precise.
- Do not treat humor or irony as neutral if it carries misogynistic meaning.
{ocr_section}
Respond ONLY in this JSON format (no markdown, no extra text):
{{
  "label": 1,
  "classification": "misogyny",
  "explanation": "Brief explanation referencing image and text with cultural context."
}}
label: 1 = misogyny, 0 = not-misogyny"""

# ── Cross-cultural (MAMI) prompt — KEY ADDITION: bridging sentence ────────────
CROSS_CULTURAL_PROMPT_TEMPLATE = """
You are a highly advanced misogyny detector, designed to analyse visual scenes
and evaluate memes for misogynistic content by examining both visual and
textual components.

You will be provided with a {language} meme image from a Western/English context.
You must evaluate this meme from the perspective of an annotator with {country}
cultural values and sensitivities. Apply {country} cultural norms and judgements
to assess whether this content would be perceived as misogynistic by someone
with an {country} cultural background, even though the meme originates from a
different cultural context.

Step-by-Step Instructions:
1. Generate a scene graph by identifying all entities, objects, attributes,
   and relationships in the scene. Include characters, expressions, text
   overlays, symbols, layout, and background elements.
2. Use the scene graph from step 1 to analyse the visual and textual content.
3. Evaluate whether an annotator from {country} would consider this content
   misogynistic based on {country} cultural values, even if the content is
   framed within Western cultural norms.

Classification Criteria:
- Classify as "misogyny" if the content directly or indirectly targets women
  or girls through: stereotyping, hatred, discrimination, sexualization,
  objectification, gender-based violence, insults, victim-blaming, mockery,
  or trivialization — even if conveyed through sarcasm or humor.
- Classify as "not-misogyny" if none of the above criteria apply.

Rules:
- Be objective, culturally aware, and precise.
- Apply {country} cultural sensitivity, not Western standards alone.
- Do not treat humor or irony as neutral if it carries misogynistic meaning.
{ocr_section}
Respond ONLY in this JSON format (no markdown, no extra text):
{{
  "label": 1,
  "classification": "misogyny",
  "explanation": "Brief explanation referencing image and text with {country} cultural context."
}}
label: 1 = misogyny, 0 = not-misogyny"""

OCR_SECTION = "\nOCR Transcription of meme text:\n\"\"\"\n{transcription}\n\"\"\"\n"

# Simplified fallback prompt — used when JSON parse fails
FALLBACK_PROMPT = """Look at this meme image carefully.
Does it contain misogynistic content that targets, demeans, stereotypes,
or discriminates against women or girls?

Reply with ONLY a single digit: 1 for misogyny, 0 for not-misogyny.
No explanation needed. Just the digit."""

# Question shown to model for each few-shot example turn
EXAMPLE_QUESTION = "Does this meme contain misogynistic content? Respond in the JSON format."


def load_pil(image_path: str):
    return Image.open(image_path).convert("RGB")


def build_prompt(language: str, country: str,
                 transcription: str = None,
                 is_cross_cultural: bool = False) -> str:
    """Select correct prompt template based on partition type."""
    template = CROSS_CULTURAL_PROMPT_TEMPLATE if is_cross_cultural \
               else NATIVE_PROMPT_TEMPLATE
    ocr = OCR_SECTION.format(transcription=transcription) if transcription else ""
    return template.format(language=language, country=country, ocr_section=ocr)


def example_answer(ex: dict) -> str:
    """
    Format the few-shot assistant answer.
    Includes the pre-written scene graph as part of reasoning
    so the model sees what a good scene graph + answer looks like.
    """
    return json.dumps({
        "scene_graph":    ex.get("scene_graph", ""),
        "label":          ex["label"],
        "classification": ex["classification"],
        "explanation":    ex["explanation"],
    }, ensure_ascii=False)


def parse_response(result: str, image_path: str) -> tuple:
    """
    Parse model JSON response.
    Returns: (classification_str, label_int, explanation_str)
    """
    if not result:
        return "not-misogyny", 0, "No response."
    try:
        # Strip markdown fences
        if "```json" in result:
            result = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            result = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            if start >= 0 and end > start:
                result = result[start:end]

        parsed = json.loads(result)
        label  = int(parsed.get("label", 0))
        return ("misogyny" if label == 1 else "not-misogyny",
                label,
                parsed.get("explanation", ""))

    except Exception as e:
        print(f"  [WARN] JSON parse failed for "
              f"{os.path.basename(image_path)}: {e}")
        # Heuristic fallback — don't fail silently
        is_m = (result is not None
                and "misogyny" in result.lower()
                and "not-misogyny" not in result.lower()
                and "not misogyny" not in result.lower())
        return ("misogyny" if is_m else "not-misogyny",
                1 if is_m else 0,
                result or "Parse failed.")


print("Prompts and utilities ready.")

Prompts and utilities ready.


In [41]:

import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit              = True,           # was load_in_8bit
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,           # nested quantisation, saves ~0.4 GiB extra
    bnb_4bit_quant_type       = "nf4",          # best quality 4-bit format
)

_hf_model     = None
_hf_processor = None
_hf_device    = "cuda" if torch.cuda.is_available() else "cpu"


def load_hf_model(model_name: str):
    global _hf_model, _hf_processor
    if _hf_model is not None:
        del _hf_model
        torch.cuda.empty_cache()

    print(f"Loading {model_name} on {_hf_device}...")
    dtype = torch.bfloat16 if _hf_device == "cuda" else torch.float32

    _hf_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        attn_implementation=get_attn_implementation(),
    ).eval()
    _hf_processor = AutoProcessor.from_pretrained(
        model_name,
        min_pixels = 128 * 28 * 28,   # was 256 * 28 * 28 — halved
        max_pixels = 512 * 28 * 28,
    )
    _hf_processor.tokenizer.padding_side = "left"
    print("Model ready.")


def get_attn_implementation():
    if not torch.cuda.is_available():
        return "eager"
    major = torch.cuda.get_device_capability()[0]
    if major >= 8:
        return "flash_attention_2"   # Ampere+ (A100, A10, RTX 3090)
    else:
        return "sdpa"


def _run_inference(messages: list, max_new_tokens: int = 512) -> str:
    """Core inference — shared by single and fallback paths."""
    from qwen_vl_utils import process_vision_info
    text         = _hf_processor.apply_chat_template(
                       messages, tokenize=False, add_generation_prompt=True)
    img_inp, vid_inp = process_vision_info(messages)
    inputs       = _hf_processor(
                       text=[text], images=img_inp, videos=vid_inp,
                       padding=True, return_tensors="pt").to(_hf_device)
    input_len    = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out_ids  = _hf_model.generate(
                       **inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return _hf_processor.decode(
               out_ids[0][input_len:], skip_special_tokens=True).strip()


def call_huggingface_single(query_image_path: str,
                            prompt: str,
                            few_shot_examples: list) -> str:
    """
    Single-image few-shot inference.
    few_shot_examples: the PARTITION-SPECIFIC list (Indian or Chinese).
    """
    messages = []
    for ex in few_shot_examples:
        messages.append({"role": "user", "content": [
            {"type": "image", "image": load_pil(ex["image_path"])},
            {"type": "text",  "text": EXAMPLE_QUESTION},
        ]})
        messages.append({"role": "assistant",
                         "content": example_answer(ex)})
    messages.append({"role": "user", "content": [
        {"type": "image", "image": load_pil(query_image_path)},
        {"type": "text",  "text": prompt},
    ]})
    return _run_inference(messages)


def call_huggingface_with_fallback(query_image_path: str,
                                   prompt: str,
                                   few_shot_examples: list) -> tuple:
    """
    Inference with automatic fallback on JSON parse failure.
    Returns: (raw_result, used_fallback: bool)

    On first attempt: full few-shot + scene graph prompt.
    On JSON failure: simplified binary question, no few-shot context.
    This eliminates the 3 known parse failures (15068, 16203, 15002).
    """
    raw = call_huggingface_single(query_image_path, prompt, few_shot_examples)

    # Quick check: does it look like valid JSON?
    try:
        text = raw
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].strip()
        else:
            s = text.find("{"); e = text.rfind("}") + 1
            if s >= 0 and e > s:
                text = text[s:e]
        json.loads(text)
        return raw, False   # success — no fallback needed

    except Exception:
        # JSON failed → re-run with simplified prompt (no few-shot)
        img_id = os.path.splitext(os.path.basename(query_image_path))[0]
        print(f"  [FALLBACK] Re-running simplified prompt for {img_id}")
        fallback_messages = [{"role": "user", "content": [
            {"type": "image", "image": load_pil(query_image_path)},
            {"type": "text",  "text": FALLBACK_PROMPT},
        ]}]
        fallback_raw = _run_inference(fallback_messages, max_new_tokens=16)
        # Convert single digit to JSON-like response
        digit = fallback_raw.strip()[:1]
        label = 1 if digit == "1" else 0
        synthetic = json.dumps({
            "label":          label,
            "classification": "misogyny" if label == 1 else "not-misogyny",
            "explanation":    f"Fallback classification (original parse failed). Raw: {fallback_raw[:100]}",
        })
        return synthetic, True


def call_huggingface_batch(query_image_paths: list,
                           prompts: list,
                           few_shot_examples: list) -> list:
    """
    Batched few-shot inference.
    Each item in the batch uses the same few_shot_examples set
    (already partition-specific by the time this is called).
    """
    from qwen_vl_utils import process_vision_info
    batch_texts  = []
    batch_images = []

    for query_image_path, prompt in zip(query_image_paths, prompts):
        messages = []
        for ex in few_shot_examples:
            messages.append({"role": "user", "content": [
                {"type": "image", "image": load_pil(ex["image_path"])},
                {"type": "text",  "text": EXAMPLE_QUESTION},
            ]})
            messages.append({"role": "assistant",
                             "content": example_answer(ex)})
        messages.append({"role": "user", "content": [
            {"type": "image", "image": load_pil(query_image_path)},
            {"type": "text",  "text": prompt},
        ]})

        text = _hf_processor.apply_chat_template(
                   messages, tokenize=False, add_generation_prompt=True)
        img_inp, _ = process_vision_info(messages)
        batch_texts.append(text)
        batch_images.extend(img_inp)

    inputs    = _hf_processor(
                    text=batch_texts, images=batch_images,
                    padding=True, return_tensors="pt").to(_hf_device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out_ids = _hf_model.generate(
                      **inputs, max_new_tokens=512, do_sample=False)

    return [
        _hf_processor.decode(out_ids[i][input_len:],
                             skip_special_tokens=True).strip()
        for i in range(len(query_image_paths))
    ]


print("Backend functions ready.")

Backend functions ready.


In [42]:
def norm_id(val):
    s = str(val).strip()
    return s[:-2] if s.endswith(".0") else s


def classify_image(image_path, model_name, country, language,
                   transcription=None, is_cross_cultural=False,
                   few_shot_examples=None):
    """Single-image classification with fallback."""
    if few_shot_examples is None:
        few_shot_examples = INDIAN_FEW_SHOT   # safe default

    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = build_prompt(language, country, transcription, is_cross_cultural)

    try:
        raw, used_fallback = call_huggingface_with_fallback(
            image_path, prompt, few_shot_examples)
    except Exception as e:
        return {"image_id": image_id, "label": -1,
                "classification": "error",
                "explanation": str(e), "full_response": str(e)}

    classification, label, explanation = parse_response(raw, image_path)
    return {
        "image_id":        image_id,
        "label":           label,
        "classification":  classification,
        "explanation":     explanation,
        "full_response":   raw,
        "used_fallback":   used_fallback,
    }


def classify_batch(image_paths, model_name, country, language,
                   transcriptions, is_cross_cultural=False,
                   few_shot_examples=None):
    """Batch classification — no fallback (batch mode)."""
    if few_shot_examples is None:
        few_shot_examples = INDIAN_FEW_SHOT

    prompts = [
        build_prompt(language, country,
                     transcriptions.get(
                         os.path.splitext(os.path.basename(p))[0]),
                     is_cross_cultural)
        for p in image_paths
    ]
    try:
        raws = call_huggingface_batch(
            image_paths, prompts, few_shot_examples)
    except Exception as e:
        return [{"image_id": os.path.splitext(os.path.basename(p))[0],
                 "label": -1, "classification": "error",
                 "explanation": str(e), "full_response": str(e),
                 "used_fallback": False}
                for p in image_paths]

    results = []
    for img_path, raw in zip(image_paths, raws):
        image_id = os.path.splitext(os.path.basename(img_path))[0]
        classification, label, explanation = parse_response(raw, img_path)
        results.append({
            "image_id":       image_id,
            "label":          label,
            "classification": classification,
            "explanation":    explanation,
            "full_response":  raw,
            "used_fallback":  False,
        })
    return results


def batch_classify(partition_cfg, model_name, output_dir):
    """
    Main classification loop for one partition.
    Automatically selects the correct few-shot set and prompt type.
    """
    image_dir        = partition_cfg["image_dir"]
    csv_path         = partition_cfg.get("csv")
    country          = partition_cfg["country"]
    language         = partition_cfg["language"]
    desc             = partition_cfg["description"]
    partition_key    = partition_cfg["label_col"]
    is_cross_cultural = partition_cfg.get("is_cross_cultural", False)

    # ── Select partition-appropriate few-shot set ─────────────────────────
    # Derive partition name from label_col
    partition_name = next(
        (k for k, v in ALL_PARTITIONS.items()
         if v["label_col"] == partition_key and v["country"] == country),
        None
    )
    few_shot_examples = PARTITION_FEW_SHOT.get(
        partition_name,
        INDIAN_FEW_SHOT if country == "India" else CHINESE_FEW_SHOT
    )

    # ── Load transcriptions ───────────────────────────────────────────────
    transcriptions = {}
    if csv_path and os.path.exists(csv_path):
        df = pd.read_csv(csv_path, sep=None, engine="python")
        if "transcriptions" in df.columns and "image_id" in df.columns:
            transcriptions = {norm_id(k): str(v)
                              for k, v in zip(df["image_id"],
                                              df["transcriptions"])}
            print(f"  Loaded {len(transcriptions)} transcriptions")

    # ── Exclude ALL few-shot images from BOTH sets (avoid data leakage) ───
    all_few_shot_ids = {
        os.path.splitext(os.path.basename(ex["image_path"]))[0]
        for examples in [INDIAN_FEW_SHOT, CHINESE_FEW_SHOT]
        for ex in examples
    }

    image_files = sorted([
        f for ext in ["*.jpg", "*.jpeg", "*.png"]
        for f in glob.glob(os.path.join(image_dir, ext))
        if os.path.splitext(os.path.basename(f))[0] not in all_few_shot_ids
    ])

    #image_files = image_files[:30]

    if not image_files:
        print(f"  [WARN] No images found in {image_dir} — skipping.")
        return []

    print(f"\n{'='*60}")
    print(f"  Partition   : {desc}")
    print(f"  Model       : {model_name}")
    print(f"  Country     : {country}  |  Language: {language}")
    print(f"  Cross-cult. : {is_cross_cultural}")
    print(f"  Images      : {len(image_files)} "
          f"(excluded {len(all_few_shot_ids)} few-shot)")
    print(f"  Few-shot    : {len(few_shot_examples)} "
          f"({'Indian' if country == 'India' else 'Chinese'} lens)")
    print(f"  Batch size  : {BATCH_SIZE}")
    print(f"{'='*60}")

    os.makedirs(output_dir, exist_ok=True)
    safe_model   = model_name.replace("/", "_").replace(":", "-")
    out_base     = os.path.join(output_dir, f"{safe_model}__{partition_key}")
    json_out     = f"{out_base}.json"
    submit_out   = f"{out_base}_submission.csv"

    results      = []
    fallback_count = 0

    # Process in batches
    for batch_start in tqdm(range(0, len(image_files), BATCH_SIZE),
                            desc=f"{model_name.split('/')[-1][:20]} | {country}",
                            unit="batch", colour="green"):
        batch_paths = image_files[batch_start:batch_start + BATCH_SIZE]

        if BATCH_SIZE == 1:
            # Single-image path — has fallback support
            result = classify_image(
                image_path       = batch_paths[0],
                model_name       = model_name,
                country          = country,
                language         = language,
                transcription    = transcriptions.get(
                    os.path.splitext(os.path.basename(batch_paths[0]))[0]),
                is_cross_cultural = is_cross_cultural,
                few_shot_examples = few_shot_examples,
            )
            if result.get("used_fallback"):
                fallback_count += 1
            results.append(result)
        else:
            # Batch path
            batch_results = classify_batch(
                image_paths      = batch_paths,
                model_name       = model_name,
                country          = country,
                language         = language,
                transcriptions   = transcriptions,
                is_cross_cultural = is_cross_cultural,
                few_shot_examples = few_shot_examples,
            )
            results.extend(batch_results)

    # ── Save outputs ──────────────────────────────────────────────────────
    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    with open(submit_out, "w", newline="") as f:
        writer = csv_module.writer(f)
        writer.writerow(["image_id", "label"])
        for r in results:
            writer.writerow([r["image_id"], r["label"]])

    total    = len(results)
    misogyny = sum(1 for r in results if r["label"] == 1)
    not_m    = sum(1 for r in results if r["label"] == 0)
    errors   = sum(1 for r in results if r["label"] == -1)

    print(f"\n  Results -> {json_out}")
    print(f"  Submit  -> {submit_out}")
    print(f"  Summary : total={total}  misogyny={misogyny}  "
          f"not-misogyny={not_m}  errors={errors}  "
          f"fallbacks={fallback_count}")

    return results


print("All functions ready.")

All functions ready.


In [43]:
# ── Run ────────────────────────────────────────────────────────────────────────

prev_model = None

for model_name in SELECTED_MODELS:
    if model_name != prev_model:
        load_hf_model(model_name)
        prev_model = model_name

    for partition_name, partition_cfg in PARTITIONS.items():
        out_dir = os.path.join(OUTPUT_DIR, partition_name)
        batch_classify(
            partition_cfg = partition_cfg,
            model_name    = model_name,
            output_dir    = out_dir,
        )

print("\nAll runs complete.")

Loading Qwen/Qwen2.5-VL-7B-Instruct on cuda...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model ready.
  Loaded 1000 transcriptions

  Partition   : MAMI — Indian perception (test)
  Model       : Qwen/Qwen2.5-VL-7B-Instruct
  Country     : India  |  Language: English
  Cross-cult. : False
  Images      : 1000 (excluded 20 few-shot)
  Few-shot    : 10 (Indian lens)
  Batch size  : 2


...

  [WARN] JSON parse failed for 15483.jpg: Expecting ',' delimiter: line 4 column 178 (char 225)


Qwen2.5-VL-7B-Instru | India:  52%|█████▏    | 260/500 [49:00<1:20:30, 20.13s/batch]

  [WARN] JSON parse failed for 15776.jpg: Unterminated string starting at: line 4 column 18 (char 69)


Qwen2.5-VL-7B-Instru | India: 100%|██████████| 500/500 [1:33:29<00:00, 11.22s/batch]


  Results -> /content/results/mami_indian/Qwen_Qwen2.5-VL-7B-Instruct__indian_labels.json
  Submit  -> /content/results/mami_indian/Qwen_Qwen2.5-VL-7B-Instruct__indian_labels_submission.csv
  Summary : total=1000  misogyny=633  not-misogyny=367  errors=0  fallbacks=0

All runs complete.


In [44]:

def score_partition(results_json_path: str, label_csv_path: str,
                    label_col: str) -> dict:
    """
    Compute Macro-F1 and Accuracy against ground truth.
    Only works when running on dev (not test — test has no labels).
    """
    from sklearn.metrics import f1_score, accuracy_score

    with open(results_json_path) as f:
        results = json.load(f)

    df = pd.read_csv(label_csv_path, sep=None, engine="python")
    df["image_id"] = df["image_id"].apply(norm_id)

    pred_map = {r["image_id"]: r["label"]
                for r in results if r["label"] != -1}

    y_true, y_pred = [], []
    for _, row in df.iterrows():
        iid = norm_id(row["image_id"])
        if iid in pred_map and pd.notna(row.get(label_col)):
            y_true.append(norm_label(row[label_col]))
            y_pred.append(pred_map[iid])

    if not y_true:
        print("  [WARN] No matching image IDs found for scoring.")
        return {}

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    accuracy = f1_score(y_true, y_pred, average="macro")  # reusing for clarity
    accuracy = __import__("sklearn.metrics", fromlist=["accuracy_score"]) \
               .accuracy_score(y_true, y_pred)

    print(f"\n  Macro-F1 : {macro_f1:.4f}")
    print(f"  Accuracy : {accuracy:.4f}")
    print(f"  Samples  : {len(y_true)}")

    return {"macro_f1": macro_f1, "accuracy": accuracy, "n": len(y_true)}


def norm_label(val) -> int:
    """Convert label to int regardless of format."""
    if isinstance(val, (int, float)):
        return int(val)
    s = str(val).strip().lower()
    if s in ("1", "misogyny"):
        return 1
    if s in ("0", "not-misogyny", "not misogyny"):
        return 0
    raise ValueError(f"Unrecognised label value: {val!r}")

In [45]:
label_col = 'indian_labels'
partition = 'mami_indian'
score_partition(
        results_json_path = f"/content/results/{partition}/Qwen_Qwen2.5-VL-7B-Instruct__{label_col}.json",
        label_csv_path    = f"{MAMI_DIR}/test_with_labels_mami.csv",
        label_col         = label_col,
    )


  Macro-F1 : 0.6459
  Accuracy : 0.6460
  Samples  : 1000


{'macro_f1': 0.6459093527943154, 'accuracy': 0.646, 'n': 1000}

In [46]:
# ── List generated submission files ────────────────────────────────────────────
import glob as _glob

sub_files = _glob.glob(os.path.join(OUTPUT_DIR, "**", "*_submission.csv"), recursive=True)
print(f"Submission files generated: {len(sub_files)}")
for f in sorted(sub_files):
    import pandas as _pd
    df = _pd.read_csv(f)
    print(f"  {os.path.basename(f):60s}  rows={len(df)}")

Submission files generated: 2
  Qwen_Qwen2.5-VL-7B-Instruct__chinese_labels_submission.csv    rows=10
  Qwen_Qwen2.5-VL-7B-Instruct__indian_labels_submission.csv     rows=1000


In [47]:
# ── Download all results as zip ───────────────────────────────────────────────
import shutil
from google.colab import files

zip_path = "/content/cc_mmd_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(f"{zip_path}.zip")
print("Downloaded cc_mmd_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded cc_mmd_results.zip
